In [22]:
import os
import urllib.request

Tokenization

In [23]:
with open("the_verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read();

In [24]:
len(raw_text)

20479

In [25]:
import re

In [26]:
tokens = re.findall(r"\w+|[^\w\s]", raw_text)
len(tokens)

4827

In [27]:
uniqueTokens = set(tokens)
len(uniqueTokens)

1148

In [28]:
allTokens = sorted(list(uniqueTokens))
allTokens.extend(["<|unk|>", "<|endoftext|>"])
vocab = {s:i for i, s in enumerate(sorted(allTokens))}
vocab

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '-': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '<|endoftext|>': 10,
 '<|unk|>': 11,
 '?': 12,
 'A': 13,
 'Ah': 14,
 'Among': 15,
 'And': 16,
 'Are': 17,
 'Arrt': 18,
 'As': 19,
 'At': 20,
 'Be': 21,
 'Begin': 22,
 'Burlington': 23,
 'But': 24,
 'By': 25,
 'Carlo': 26,
 'Chicago': 27,
 'Claude': 28,
 'Come': 29,
 'Croft': 30,
 'Destroyed': 31,
 'Devonshire': 32,
 'Don': 33,
 'Dubarry_': 34,
 'Emperors': 35,
 'Florence': 36,
 'For': 37,
 'Gallery': 38,
 'Gideon': 39,
 'Gisburn': 40,
 'Gisburns': 41,
 'Grafton': 42,
 'Greek': 43,
 'Grindle': 44,
 'Grindles': 45,
 'HAD': 46,
 'Had': 47,
 'Hang': 48,
 'Has': 49,
 'He': 50,
 'Her': 51,
 'Hermia': 52,
 'His': 53,
 'How': 54,
 'I': 55,
 'If': 56,
 'In': 57,
 'It': 58,
 'Jack': 59,
 'Jove': 60,
 'Just': 61,
 'Lord': 62,
 'Made': 63,
 'Miss': 64,
 'Money': 65,
 'Monte': 66,
 'Moon': 67,
 'Mr': 68,
 'Mrs': 69,
 'My': 70,
 'Never': 71,
 'No': 72,
 'Now': 73,
 'Nutley': 74,
 'Of': 75,
 'Oh': 76,
 'On': 77

In [29]:
class SimpleTokenizer:

    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i : s for s, i in vocab.items()}

    def encode(self, text):
        tokens = re.findall(r"\w+|[^\w\s]", text)
        tokens = [item if item in self.str_to_int else "<|unk|>" for item in tokens]
        ids = [self.str_to_int[s] for s in tokens]
        return ids
    
    def decode(self, ids):
        tokens = [self.int_to_str[i] for i in ids]
        text = " ".join(tokens)
        #replace spaces before punctuations
        text = re.sub(r"\s+([.,!?;:'])", r"\1", text)
        return text


In [30]:
tokenizer = SimpleTokenizer(vocab)
text = "'a, merely! he's aes'thetic satisfaction'"
res = tokenizer.encode(text)
tokenizer.decode(res)

"' a, merely! he' s <|unk|>' <|unk|> satisfaction'"

Byte Pair Encoding

In [31]:
import tiktoken

In [32]:
tokenizer = tiktoken.get_encoding("gpt2")


In [33]:
text = "Hellotherewhoareyou <|endoftext|> no ma'am"
tokenizer.encode(text, allowed_special={"<|endoftext|>"})

[28254, 313, 1456, 8727, 533, 5832, 220, 50256, 645, 17266, 6, 321]

In [34]:
tokenizer.decode([28254, 313, 1456, 8727, 533, 5832, 220, 50256, 645, 17266, 6, 321])

"Hellotherewhoareyou <|endoftext|> no ma'am"

Data Sampling

In [35]:
import torch
from torch.utils.data import Dataset, DataLoader;

In [36]:
class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        tokens = tokenizer.encode(text, allowed_special = {"<|endoftext|>"})

        for i in range(0, len(tokens) - max_length, stride):
            input_chunk = tokens[i: i + max_length]
            target_chunk = tokens[i + 1: i + 1 + max_length]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [37]:
def create_dataloader(text, batch_size = 4, max_length = 256, stride=128, shuffle = True, drop_last=True, n_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(text, tokenizer, max_length, stride)


    dataloader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=shuffle,  num_workers=n_workers, drop_last=drop_last )

    return dataloader

In [ ]:
dataloader = create_dataloader(raw_text, 4, 4, 4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs: ", inputs)
print("Target: ", targets)

Inputs:  tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257]])
Target:  tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922]])


tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922]])

Token Embeddings

In [44]:
embedding_layer = torch.nn.Embedding(tokenizer.n_vocab, 3)
embedding_layer.weight

Parameter containing:
tensor([[-0.9661,  1.7479, -1.5042],
        [ 0.7693,  0.9534, -0.2806],
        [-0.9529, -0.6787,  2.2594],
        ...,
        [ 0.0282,  1.1477, -0.4558],
        [-1.0267, -3.0680, -1.4235],
        [ 1.1870,  0.2324, -1.6716]], requires_grad=True)

In [46]:
embeddings = embedding_layer(inputs)
embeddings.shape()

TypeError: 'torch.Size' object is not callable